In [1]:
import nltk
from nltk.stem.lancaster import LancasterStemmer
stemmer=LancasterStemmer()

import numpy
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import random
import json
import pickle

In [2]:
with open("intents.json") as file:
    data=json.load(file)

In [3]:
print(data["intents"])

[{'tag': 'greeting', 'patterns': ['Hi', 'How are you', 'Is anyone there?', 'Hello', 'Good day', 'Whats up'], 'responses': ['Hello!', 'Good to see you again!', 'Hi there, how can I help?'], 'context_set': ''}, {'tag': 'goodbye', 'patterns': ['cya', 'See you later', 'Goodbye', 'I am Leaving', 'Have a Good day'], 'responses': ['Sad to see you go :(', 'Talk to you later', 'Goodbye!'], 'context_set': ''}, {'tag': 'age', 'patterns': ['how old', 'how old is prakriti', 'what is your age', 'how old are you', 'age?'], 'responses': ['I am 19 years old!', '19 years young!'], 'context_set': ''}, {'tag': 'name', 'patterns': ['what is your name', 'what should I call you', 'whats your name?'], 'responses': ['You can call me Prakriti.', "I'm Prakriti!", "I'm Prakriti aka Prakriti Bhandari."], 'context_set': ''}, {'tag': 'shop', 'patterns': ['Id like to buy something', 'whats on the menu', 'what do you reccommend?', 'could i get something to eat'], 'responses': ['We sell chocolate chip cookies for $2!',

In [4]:
try:
    with open("data.pickle","rb") as f:
        words,labels,training,output=pickle.load(f)
except:
    words=[]
    labels=[]
    docs_x=[]
    docs_y=[]
    for intent in data["intents"]:
        for pattern in intent["patterns"]:
            wrds=nltk.word_tokenize(pattern)
            words.extend(wrds)
            docs_x.append(wrds)
            docs_y.append(intent["tag"])

        if intent["tag"] not in labels:
            labels.append(intent["tag"])
            
    words=[stemmer.stem(w.lower()) for w in words if w!= "?"]
    words=sorted(list(set(words)))
    
    labels=sorted(labels)
    
    training=[]
    output=[]
    
    out_empty=[0 for _ in range(len(labels))]

    for x,doc in enumerate(docs_x):
        bag=[]

        wrds=[stemmer.stem(w.lower()) for w in doc]

        for w in words:
            if w in wrds:
                bag.append(1)
            else:
                bag.append(0)
                
        output_row=out_empty[:]
        output_row[labels.index(docs_y[x])]=1
        
        training.append(bag)
        output.append(output_row)

# random=numpy.array(training)
# output=np.array(output)
            

In [5]:
training=numpy.array(training)
output=numpy.array(output)

with open("data.pickle","wb") as f:
        pickle.dump((words,labels,training,output), f)

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

model.add(Dense(8,
                input_shape=(len(training[0]),),
                activation="relu"))

model.add(Dense(8,
                activation="relu"))

model.add(Dense(len(output[0]),
                activation="softmax"))

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

C:\Users\LENOVO\Documents\chatbot-project\chatbot-env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [8]:
import sys
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [9]:
try:
    model.load("model.tflearn")
except:
    history = model.fit(
    training,
    output,
    epochs=1000,
    batch_size=8,
    verbose=1
)
    model.save("chatbot_model.keras")

Epoch 1/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.2308 - loss: 1.7965
Epoch 2/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2308 - loss: 1.7886
Epoch 3/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2692 - loss: 1.7820 
Epoch 4/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.2692 - loss: 1.7759 
Epoch 5/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2692 - loss: 1.7708 
Epoch 6/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2692 - loss: 1.7645 
Epoch 7/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3077 - loss: 1.7591 
Epoch 8/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3077 - loss: 1.7541
Epoch 9/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3077 - loss: 1.7487 
Epoch 10/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3077 - loss: 1.7436 
Epoch 11/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3077 - loss: 1.7388 
Epoch 12/1000
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - a

In [10]:
def bag_of_words(s,words):
    bag=[0 for _ in range(len(words))]

    s_words=nltk.word_tokenize(s)
    s_words=[stemmer.stem(word.lower()) for word in s_words]

    for se in s_words:
        for i,w in enumerate(words):
            if w==se:
                bag[i]=1
                
    return numpy.array(bag)        

In [ ]:
def chat():
    print("Start talking with the bot(type quit to stop)!")
    while True:
        inp=input("You: ")
        if inp.lower()=="quit":
            break
            
        results=model.predict(bag_of_words(inp,words).reshape(1,-1))[0]
        results_index=numpy.argmax(results)
        tag=labels[results_index]
        if results[results_index]>0.7:
            for tg in data["intents"]:
                if tg['tag']==tag:
                    responses=tg['responses']
            print(random.choice(responses))     
        else:
            print("I didn't get that,try again.")

chat()


Start talking with the bot(type quit to stop)!


You:  Hi


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
Hi there, how can I help?


You:  How are you doing these days?


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Hi there, how can I help?


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
I didn't get that,try again.


You:  bye


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
I didn't get that,try again.


You:  How are you Prakriti?


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Hello!


In [ ]:
import os
for f in ["data.pickle", "model.tflearn", "chatbot_model.keras"]:
    if os.path.exists(f):
        os.remove(f)

In [ ]:
import os
if os.path.exists("chatbot_model.keras"):
    model = tf.keras.models.load_model("chatbot_model.keras")
else:
    history = model.fit(
        training,
        output,
        epochs=1000,
        batch_size=8,
        verbose=1
    )
    model.save("chatbot_model.keras")